In [1]:
GEMINI_1_5_FLASH_CONFIG = {
# Total dictionary size: number of unique tokens (words/subwords) the model recognizes
    "vocab_size": 256000,     
    # Context window: maximum number of tokens the model can look at in a single sequence
    "context_length": 1024, 
    # Embedding dimension: vector size used to represent each token in hidden memory space
    "emb_dim": 256,            
    # Attention heads: number of parallel "viewpoints" focusing on token relationships
    "n_heads": 8,              
    # Key/Value heads: number of KV heads (8 means standard Multi-Head Attention; fewer means Grouped-Query Attention)
    "n_kv_heads": 8, 
    # Depth: number of stacked Transformer blocks/layers in the network
    "n_layers": 4,         
    # Model design: causal auto-regressive decoder where all layers participate equally
    "architecture_type": "dense_decoder", 
    # Dropout rate: percentage (0.0 = 0%) of random neurons turned off during training to prevent overfitting
    "drop_rate": 0.0,
    # Bias terms: whether to add extra learnable offset vectors to Query, Key, and Value projections
    "qkv_bias": False,
    # Activation function: non-linear math operation used in the feed-forward layers (Swish-Gated Linear Unit)
    "activation": "SwiGLU",
    # MLP hidden size: inner expansion width of the feed-forward network in each layer
    "mlp_dim": 682,
}

[ Input Embedding Matrix: x ]
                             │
              ┌──────────────┴──────────────┐
              ▼                             ▼
          ( x * cos )                _rotate_half(x)
              │                             │
              │                             ▼
              │                     ( flipped_x * sin )
              │                             │
              └──────────────┬──────────────┘
                             ▼
                    [ Added Together ]
                             │
                             ▼
                 [ Rotated Token Embeddings ]

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RotatoryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, theta=10000.0):
        super().__init__()
        self.dim = dim # dim = head_dim
        #dim = 64, so generate even number series till 64
        inv_freq = 1.0/(theta ** (torch.arange(0, dim, 2).float() / dim))#theta_i = theta^(-2i/d)
        self.register_buffer("inv_freq", inv_freq, persistent=False)# Store buffer in memory

        # Precompute text position indices
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)# torch.outer computes the outer product of two 1D vectors
        
        # Separate out absolute coordinate mappings for cosine and sine transformations
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)#emb.cos() = cos(m*theta_i)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def _rotate_half(self, x):
        x1 = x[..., :self.dim // 2]# ROTATE 180 FOR EXTRACTING FIRST HALF OF VECTOR DIMENSION
        x2 = x[..., self.dim // 2:]# ROTATE 180 FOR EXTRACTING SECOND HALF OF VECTOR DIMENSION
        return torch.cat((-x2, x1), dim=-1) #Swap halves and negate second half for 2D rotation: [-x2, x1]

    def forward(self, x, seq_len):
        # x shape: [batch_size, n_heads, seq_len, head_dim]
            cos = self.cos_cached[:seq_len, :].unsqueeze(0).unsqueeze(1) # [1, 1, seq_len, head_dim]
            sin = self.sin_cached[:seq_len, :].unsqueeze(0).unsqueeze(1)
        
        # # Apply 2D rotation formula: (x * cos) + (rotate_half(x) * sin)

            return (x * cos) + (self._rotate_half(x) * sin)

In [3]:
class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.emb_dim = cfg["emb_dim"] # Total embedding dimension
        self.n_heads = cfg["n_heads"]# Number of Query attention heads
        self.n_kv_heads = cfg["n_kv_heads"]# Number of Key/Value heads
        self.head_dim = self.emb_dim // self.n_heads

        self.num_queries_per_kv = self.n_heads // self.n_kv_heads

         # Linear projections for Query, Key, Value, and Output VECTORS
        self.q_proj = nn.Linear(self.emb_dim, self.n_heads * self.head_dim, bias=cfg["qkv_bias"])
        self.k_proj = nn.Linear(self.emb_dim, self.n_kv_heads * self.head_dim, bias=cfg["qkv_bias"])
        self.v_proj = nn.Linear(self.emb_dim, self.n_kv_heads * self.head_dim, bias=cfg["qkv_bias"])
        self.out_proj = nn.Linear(self.n_heads * self.head_dim, self.emb_dim, bias=False)

         # Instantiate RoPE embedding using per-head dimension
        self.rope = RotatoryEmbedding(dim=self.head_dim, max_seq_len=cfg["context_length"])

    def forward(self, x):

        # b = batch_size
        # s = sequence_length = No.of tokens in sequence
        # c = channels = hidden feature dimensions that store context
        b, s, c = x.shape
        q = self.q_proj(x).view(b, s, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, s, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, s, self.n_kv_heads, self.head_dim).transpose(1, 2)

        #implementation RoPE to Query and Key Tesnor
        q = self.rope(q, seq_len=s)
        k = self.rope(k, seq_len=s)
        
        # Expand Key/Values to match Query groups if using strict GQA structures
        if self.num_queries_per_kv > 1:
            k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
            v = v.repeat_interleave(self.num_queries_per_kv, dim=1)

        # Create upper triangular matrix filled with -infinity 
        mask = torch.triu(torch.full((s, s), float('-inf'), device=x.device), diagonal=1)
        # Compute scaled dot-product attention scores: (Q * K^T) / sqrt(head_dim)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        # Casual Masking to Weights for training
        attn_weights = attn_weights + mask.unsqueeze(0).unsqueeze(1)
        #Implementation of Softmax
        attn_weights = F.softmax(attn_weights, dim=-1)

        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).contiguous().view(b, s, -1)
        
        return self.out_proj(context)

In [4]:
class GeminiSwiGLU(nn.Module):
    """
    Modern 3-matrix gate architecture scaling out across calculated mlp dimensions.
    """
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg["emb_dim"], cfg["mlp_dim"], bias=False)
        self.w_up   = nn.Linear(cfg["emb_dim"], cfg["mlp_dim"], bias=False)
        self.w_down = nn.Linear(cfg["mlp_dim"], cfg["emb_dim"], bias=False)

    def forward(self, x):
        # Element-wise gating computation: Swish(Gate) * Up-Projection using Sigmoid Linear Unit(F.SILU) activation function
        
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

In [5]:
class DummyRMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim)) # Learnable scaling parameter (gamma)


    def forward(self, x):
       # Compute mean squared value across hidden dimension
        variance = x.pow(2).mean(-1, keepdim=True)
        # Normalize: x / sqrt(variance + epsilon)
        x_normed = x * torch.rsqrt(variance + self.eps)
        # Scale by learnable weights
        return x_normed * self.weight

In [6]:
class DummyGeminiFlashBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Norm instances for incoming token routing
        self.attn_norm = DummyRMSNorm(cfg["emb_dim"])
        self.ffn_norm  = DummyRMSNorm(cfg["emb_dim"])
        
        # Sub-layer initializations
        self.attn = GQA(cfg)
        self.ffn  = GeminiSwiGLU(cfg)

    def forward(self, x):
        # GEMINI PARALLEL ROUTING EXECUTION:
        # Both computation streams branch simultaneously out from the structural input x.
        residual_attn = self.attn(self.attn_norm(x))
        residual_ffn  = self.ffn(self.ffn_norm(x))
        
        # Combine input residual x with both concurrent sub-layer results
        return x + residual_attn + residual_ffn


In [7]:


class DummyGemini15FlashModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # 1. Native Multimodal Tokenizer Embedding space
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])

        #No postional encoding due as it is directly implemented inside attention layer using RoPE(Rotatory posiitional encoding)

        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[DummyGeminiFlashBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = DummyRMSNorm(cfg["emb_dim"])

        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape

        x = self.tok_emb(in_idx) #create vector embeddings of tokens
        x = self.drop_emb(x) #dropout embedding layer to  intoduce variation in training patterns and reduce overfitting
        x = self.trf_blocks(x)#trnasformers block
        x = self.final_norm(x)#RMS Normalizaion layer
        logits = self.out_head(x)#Prediction Score for every single words
        return logits

In [8]:
from transformers import AutoTokenizer
from dotenv import load_dotenv
import os

load_dotenv()

# 2. Retrieve the token safely
hf_token = os.getenv("HF_ACCESS_TOKEN")

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b", token = hf_token)

/var/www/html/llm_scratch/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[    2,  9112,  8395, 14574,   692],
        [    2,  9112,  1744, 12723,   476]])


In [10]:
torch.manual_seed(123)
model = DummyGemini15FlashModel(GEMINI_1_5_FLASH_CONFIG)
logits = model(batch)
print("Output shape:([b,s,vocab_score]):", logits.shape)
print(logits)

#For each token position in the sentence, the model assigns a raw score to all 256,000 words in its vocabulary
#A higher score means the model believes that word is a more likely candidate for the next token.
#A negative or low score means the model considers that word unlikely

total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")


Output shape:([b,s,vocab_score]): torch.Size([2, 5, 256000])
tensor([[[ 0.4045, -0.1929,  1.0951,  ...,  0.5795,  0.4946, -0.6040],
         [-0.2921, -0.1640, -0.0731,  ..., -0.3277, -0.2551,  0.3419],
         [ 0.3971,  0.3938, -0.5678,  ...,  0.1892, -0.9189,  1.4748],
         [ 0.5200, -0.4614,  0.7872,  ..., -0.2846,  0.3776,  1.2441],
         [ 0.9645,  0.1765, -1.2376,  ..., -0.1706, -0.3878,  0.6749]],

        [[ 0.4045, -0.1929,  1.0951,  ...,  0.5795,  0.4946, -0.6040],
         [-0.2921, -0.1640, -0.0731,  ..., -0.3277, -0.2551,  0.3419],
         [-0.0729, -0.6605, -0.2348,  ..., -0.2332, -0.5457, -1.1997],
         [-0.1351,  0.2707,  0.0474,  ...,  0.4101, -0.4209, -0.8386],
         [-0.7552,  0.2341, -0.2152,  ..., -0.1937, -0.7669,  1.3361]]],
       grad_fn=<UnsafeViewBackward0>)
Total number of parameters: 134,217,984


In [11]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)
total_size_bytes = total_params * 4

# Convert to megabytes
total_size_mb = total_size_bytes / (1024 * 1024)

print(f"Total size of the model: {total_size_mb:.2f} MB")

Total number of parameters: 134,217,984
Token embedding layer shape: torch.Size([256000, 256])
Output layer shape: torch.Size([256000, 256])
Total size of the model: 512.00 MB


In [12]:
probs = F.softmax(logits, dim=-1)# Apply softmax function
pred_token_ids = torch.argmax(probs, dim=-1)# Scan maximum score from 256000 probability and return token id of max score


# Extract predictions and confidence for the next word
next_tokens = pred_token_ids[:, -1]
next_confidences = probs[torch.arange(logits.size(0)), -1, next_tokens]

for idx, (tok_id, conf) in enumerate(zip(next_tokens, next_confidences)):
    word = tokenizer.decode([tok_id.item()])
    print(f"Sentence {idx+1}: Predicted Next Word = '{word}' (Token ID: {tok_id.item()}) | Confidence = {conf.item()*100:.4f}%")

# Next-token accuracy prediction against shifted targets

target_ids = batch[:, 1:]
input_pred_ids = pred_token_ids[:, :-1]
correct = (input_pred_ids == target_ids).float()
accuracy = correct.mean().item() * 100
print(f"Model Next-Token Accuracy: {accuracy:.2f}%")


Sentence 1: Predicted Next Word = '丫头' (Token ID: 148104) | Confidence = 0.0043%
Sentence 2: Predicted Next Word = ' bau' (Token ID: 36721) | Confidence = 0.0040%
Model Next-Token Accuracy: 0.00%


In [13]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]  

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [14]:
start_context = "My name is Preet Shah"

encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [2, 2926, 1503, 603, 2769, 523, 33149]
encoded_tensor.shape: torch.Size([1, 7])


In [15]:
model.eval()

DummyGemini15FlashModel(
  (tok_emb): Embedding(256000, 256)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): DummyGeminiFlashBlock(
      (attn_norm): DummyRMSNorm()
      (ffn_norm): DummyRMSNorm()
      (attn): GQA(
        (q_proj): Linear(in_features=256, out_features=256, bias=False)
        (k_proj): Linear(in_features=256, out_features=256, bias=False)
        (v_proj): Linear(in_features=256, out_features=256, bias=False)
        (out_proj): Linear(in_features=256, out_features=256, bias=False)
        (rope): RotatoryEmbedding()
      )
      (ffn): GeminiSwiGLU(
        (w_gate): Linear(in_features=256, out_features=682, bias=False)
        (w_up): Linear(in_features=256, out_features=682, bias=False)
        (w_down): Linear(in_features=682, out_features=256, bias=False)
      )
    )
    (1): DummyGeminiFlashBlock(
      (attn_norm): DummyRMSNorm()
      (ffn_norm): DummyRMSNorm()
      (attn): GQA(
        (q_proj): Linear(in_features=256, 

In [16]:

out = generate_text_simple(
    model=model,
    idx=encoded_tensor, 
    max_new_tokens=6, 
    context_size=GEMINI_1_5_FLASH_CONFIG["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[     2,   2926,   1503,    603,   2769,    523,  33149, 107354,  60272,
         170787, 130314, 104346, 249998]])
Output length: 13


In [17]:

decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

# <bos> = beginning of sequence.It is a indicate starting of sequence inserted by gemini gemma tokenizer model

<bos>My name is Preet Shah YusufQuercrying Lombard< ͯ
